# Phase 4 — Exploratory Data Analysis

## TL;DR

- Crop type is the strongest visible separator of yield: median yield ranges from about 12.3k hg/ha for sorghum to 161.9k hg/ha for potatoes.
- Overall median yield rises from 35.2k hg/ha in 1990 to 49.2k hg/ha in 2013, but this descriptive trend does not establish causation.
- Overall numeric associations with yield are weak. Pesticide use has the largest Spearman correlation (about 0.22), followed by year and rainfall (about 0.08 each), while temperature is weakly negative (about -0.11).
- Rainfall is constant through time within every country in this modelling table, so it cannot explain year-to-year yield changes within a country.
- Country yield rankings are not pure productivity rankings because countries grow different mixes of crops.


## Context & Methods

This notebook explores the validated country-crop-year modelling table created in Phase 3. It uses summary statistics, distributions, grouped medians, time trends, and Spearman correlations.

### Key assumptions

- Each row represents one country, crop, and year observation.
- Yield is measured in hectograms per hectare (`hg/ha`).
- Pesticide values are national totals in tonnes, not crop-specific application rates.
- Rainfall is country-level and constant across years in the merged table.
- Associations are descriptive and do not establish causation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/processed/crop_yield_modeling.csv"


## Data

### 1. Load the validated modelling table


In [ ]:
crop_yield = pd.read_csv(DATA_PATH)

print(f"Rows: {len(crop_yield):,}")
print(f"Columns: {crop_yield.shape[1]}")
crop_yield.head()


### 2. Reconfirm the analytical grain and coverage


In [ ]:
coverage = pd.Series({
    "countries": crop_yield["area"].nunique(),
    "crops": crop_yield["item"].nunique(),
    "first_year": crop_yield["year"].min(),
    "last_year": crop_yield["year"].max(),
    "missing_values": int(crop_yield.isna().sum().sum()),
    "duplicate_country_crop_years": int(
        crop_yield.duplicated(["area", "item", "year"]).sum()
    ),
})
coverage


## Results

### 3. Inspect numeric distributions

The median is emphasized because yield, rainfall, and especially pesticide totals are right-skewed. Large values are not automatically errors; they require domain interpretation.


In [ ]:
numeric_columns = [
    "yield_hg_per_ha",
    "average_rainfall_mm_per_year",
    "pesticides_tonnes",
    "average_temperature_c",
]

numeric_summary = crop_yield[numeric_columns].describe().T
numeric_summary["median"] = crop_yield[numeric_columns].median()
numeric_summary["skewness"] = crop_yield[numeric_columns].skew()
numeric_summary[["min", "25%", "median", "mean", "75%", "max", "skewness"]]


In [ ]:
distribution_labels = {
    "yield_hg_per_ha": "Yield (hg/ha)",
    "average_rainfall_mm_per_year": "Rainfall (mm/year)",
    "pesticides_tonnes": "Pesticides (tonnes)",
    "average_temperature_c": "Temperature (°C)",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, column in zip(axes.flat, numeric_columns):
    values = crop_yield[column]
    axis.hist(values, bins=35, color="#3B6EA8", edgecolor="white")
    axis.axvline(values.median(), color="#B9770E", linestyle="--", linewidth=2,
                 label=f"Median: {values.median():,.1f}")
    axis.set_title(distribution_labels[column])
    axis.set_ylabel("Observations")
    axis.legend()

fig.suptitle("Numeric Feature Distributions", fontsize=18, fontweight="bold")
fig.tight_layout()
plt.show()


### 4. Compare yield across crops

Fresh-weight crops such as potatoes, cassava, plantains, yams, and sweet potatoes naturally have much higher mass yields than grains and soybeans. Crop type must therefore be represented in the model.


In [ ]:
crop_summary = (
    crop_yield.groupby("item")["yield_hg_per_ha"]
    .agg(observations="size", median="median", mean="mean", minimum="min", maximum="max")
    .sort_values("median", ascending=False)
)
crop_summary


In [ ]:
fig, axis = plt.subplots(figsize=(10, 6))
crop_summary.sort_values("median")["median"].plot.barh(ax=axis, color="#3B6EA8")
axis.set_title("Median Yield by Crop")
axis.set_xlabel("Median yield (hg/ha)")
axis.set_ylabel("")
plt.tight_layout()
plt.show()


### 5. Examine the yield trend over time

The gap between 2002 and 2004 represents missing 2003 observations, not zero yield.


In [ ]:
year_summary = (
    crop_yield.groupby("year")["yield_hg_per_ha"]
    .agg(observations="size", median="median", mean="mean")
    .reindex(range(crop_yield["year"].min(), crop_yield["year"].max() + 1))
)

fig, axis = plt.subplots(figsize=(12, 5))
axis.plot(year_summary.index, year_summary["median"], marker="o", label="Median")
axis.plot(year_summary.index, year_summary["mean"], marker="s", linestyle="--", label="Mean")
axis.set_title("Yield by Year")
axis.set_xlabel("Year")
axis.set_ylabel("Yield (hg/ha)")
axis.legend()
plt.tight_layout()
plt.show()

year_summary.loc[[1990, 2002, 2003, 2004, 2013]]


### 6. Measure numeric associations with yield

Spearman correlation measures whether two variables generally move in the same or opposite direction. It does not prove that one variable causes the other.


In [ ]:
correlation_features = {
    "Rainfall": "average_rainfall_mm_per_year",
    "Pesticides": "pesticides_tonnes",
    "Temperature": "average_temperature_c",
    "Year": "year",
}

spearman_with_yield = pd.Series({
    label: crop_yield[[column, "yield_hg_per_ha"]].corr(method="spearman").iloc[0, 1]
    for label, column in correlation_features.items()
}).sort_values()
spearman_with_yield


In [ ]:
fig, axis = plt.subplots(figsize=(9, 5))
colors = ["#8FA8C7" if value < 0.20 else "#3B6EA8" for value in spearman_with_yield]
spearman_with_yield.plot.barh(ax=axis, color=colors)
axis.axvline(0, color="#1F2937", linewidth=1)
axis.set_title("Spearman Correlation with Yield")
axis.set_xlabel("Correlation coefficient")
axis.set_ylabel("")
plt.tight_layout()
plt.show()


### 7. Check the rainfall limitation

If each country has only one rainfall value across all years, rainfall can distinguish countries but cannot explain annual rainfall changes within a country.


In [ ]:
rainfall_values_per_country = (
    crop_yield.groupby("area")["average_rainfall_mm_per_year"].nunique()
)

pd.Series({
    "countries_checked": rainfall_values_per_country.size,
    "countries_with_one_rainfall_value": int(rainfall_values_per_country.eq(1).sum()),
    "countries_with_multiple_rainfall_values": int(rainfall_values_per_country.gt(1).sum()),
})


### 8. Demonstrate crop-mix confounding in country comparisons

A country's overall median yield depends partly on which crops appear in its records. The table below is descriptive and must not be interpreted as a pure ranking of agricultural productivity.


In [ ]:
country_summary = (
    crop_yield.groupby("area")
    .agg(
        observations=("yield_hg_per_ha", "size"),
        crops=("item", "nunique"),
        median_yield=("yield_hg_per_ha", "median"),
    )
    .query("observations >= 50")
    .sort_values("median_yield", ascending=False)
)
country_summary.head(10)


## Checks


In [ ]:
assert len(crop_yield) == 13_130
assert crop_yield["area"].nunique() == 101
assert crop_yield["item"].nunique() == 10
assert not crop_yield.duplicated(["area", "item", "year"]).any()
assert 2003 not in crop_yield["year"].unique()
assert rainfall_values_per_country.eq(1).all()
assert crop_summary.index[0] == "Potatoes"
assert crop_summary.index[-1] == "Sorghum"
assert np.isclose(spearman_with_yield["Pesticides"], 0.2201, atol=0.001)
print("All Phase 4 EDA checks passed.")


## Takeaways

1. Crop type must be included because yield levels differ substantially by crop.
2. The time trend is useful for prediction, but it may represent technology, improved varieties, management, crop composition, or other unobserved changes.
3. Pesticide use has only a weak positive overall association with yield and is measured at national rather than crop level; it must not be interpreted causally.
4. Static rainfall values limit the model's ability to learn annual weather effects.
5. Country-level train/test splitting or time-aware validation should be considered to avoid overly optimistic results from closely related observations.

## Next steps

- Choose a leakage-resistant train/validation/test strategy.
- encode country and crop categories inside a reproducible preprocessing pipeline;
- establish a simple regression baseline before testing more complex models;
- compare performance against a naive median prediction.
